# Beschreibung: 

# Importe:

In [1]:
import sys
import os

import pandas as pd
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, repo_root)

from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules, compute_coverage

print(f"Pfad zu allen Daten: {repo_root} \nPfad dieser Datei: {os.getcwd()}")

Pfad zu allen Daten: /home/samel/01. Projekte/01. Master/COMPARE_RST 
Pfad dieser Datei: /home/samel/01. Projekte/01. Master/COMPARE_RST/Manuelle_Ausfuehrungen/Heart_Failure_Prediction


# Daten laden:
Heart Failure Prediction Dataset: https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data?select=heart_failure_clinical_records_dataset.csv

In [2]:
heart = pd.read_csv(f"{repo_root}/Daten/health/Heart_Failure_Prediction.csv")
print(heart.shape)

(299, 13)


# Ausführung:

### Vorbereitung: (Datenaufbereitung)

In [3]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "DEATH_EVENT" # Macht aus dem Kontext heraus am meisten Sinn(Vor allem wenn es sich darum dreht)

In [4]:
cutoffs = {
    "Age": [40, 55, 65],      # jung <40, mittel, erhöht, 65<= hoch
}

# Diskretisierung der numerischen Daten:
X = heart.drop(columns=[decision_attr])

# ALLE Konditionsattribute diskretisieren (inkl. kategoriale)
heart_disc = discretize(X, bins=5, cutoffs=cutoffs)

# Entscheidungsattribut wieder anhängen
heart_disc[decision_attr] = heart[decision_attr]
print(heart_disc.columns)

Index(['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
       'ejection_fraction', 'high_blood_pressure', 'platelets',
       'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time',
       'age_disc', 'anaemia_disc', 'creatinine_phosphokinase_disc',
       'diabetes_disc', 'ejection_fraction_disc', 'high_blood_pressure_disc',
       'platelets_disc', 'serum_creatinine_disc', 'serum_sodium_disc',
       'sex_disc', 'smoking_disc', 'time_disc', 'DEATH_EVENT'],
      dtype='object')


In [5]:
cond_attrs = []
for col in heart_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte
        elif heart_disc[col].dtype == "object":
            cond_attrs.append(col)
print(cond_attrs)

['age_disc', 'anaemia_disc', 'creatinine_phosphokinase_disc', 'diabetes_disc', 'ejection_fraction_disc', 'high_blood_pressure_disc', 'platelets_disc', 'serum_creatinine_disc', 'serum_sodium_disc', 'sex_disc', 'smoking_disc', 'time_disc']


### Datenbetrachtung:

In [6]:
reduct, info = quick_reduct(heart_disc, cond_attrs, decision_attr)
rules = induce_rules(heart_disc, reduct, decision_attr)

γ(C) mit allen Attributen: 1.000000
Einzel-γ-Werte:
  age_disc: γ = 0.000000
  anaemia_disc: γ = 0.000000
  creatinine_phosphokinase_disc: γ = 0.000000
  diabetes_disc: γ = 0.000000
  ejection_fraction_disc: γ = 0.000000
  high_blood_pressure_disc: γ = 0.000000
  platelets_disc: γ = 0.000000
  serum_creatinine_disc: γ = 0.010033
  serum_sodium_disc: γ = 0.000000
  sex_disc: γ = 0.000000
  smoking_disc: γ = 0.000000
  time_disc: γ = 0.000000

Mindestens ein Attribut hat γ({a}) > 0 → benutze quick_reduct_monotone.


# Resultate

In [7]:
print("Ergebnisse:")

print(f"\n{decision_attr}: {reduct}")
print(f"\nAnzahl Regeln: {len(rules)}\n")

for r in rules[:10]:
    print(r)

#print("Rules:", rules_pass_biased[2]) # Einzeln
#print("Rules:", rules_pass_biased) # Das wären alle

Ergebnisse:

DEATH_EVENT: ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'age_disc', 'high_blood_pressure_disc']

Anzahl Regeln: 290

{'premise': {'serum_creatinine_disc': np.int64(0), 'time_disc': np.int64(0), 'ejection_fraction_disc': np.int64(0), 'creatinine_phosphokinase_disc': np.int64(3), 'platelets_disc': np.int64(2), 'age_disc': np.int64(4), 'high_blood_pressure_disc': np.int64(1)}, 'decision': np.int64(1), 'support': 1}
{'premise': {'serum_creatinine_disc': np.int64(0), 'time_disc': np.int64(0), 'ejection_fraction_disc': np.int64(0), 'creatinine_phosphokinase_disc': np.int64(3), 'platelets_disc': np.int64(2), 'age_disc': np.int64(3), 'high_blood_pressure_disc': np.int64(1)}, 'decision': np.int64(1), 'support': 1}
{'premise': {'serum_creatinine_disc': np.int64(0), 'time_disc': np.int64(0), 'ejection_fraction_disc': np.int64(0), 'creatinine_phosphokinase_disc': np.int64(3), 'platelets_disc': np.int64(0), 'age_d

In [8]:
listen = [
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'age_disc', 'high_blood_pressure_disc'],
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'age_disc'],
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'high_blood_pressure_disc'], 
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'high_blood_pressure_disc'],
    ['time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'high_blood_pressure_disc'], 
    ['serum_creatinine_disc', 'time_disc', 'creatinine_phosphokinase_disc', 'platelets_disc', 'age_disc', 'high_blood_pressure_disc'],
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'platelets_disc'],
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc'],
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'platelets_disc', 'age_disc', 'high_blood_pressure_disc'],
    ['serum_creatinine_disc', 'time_disc', 'ejection_fraction_disc', 'creatinine_phosphokinase_disc', 'age_disc', 'high_blood_pressure_disc']
    ]

i = 0
for liste in listen:
    i += 1
    print(f"{i}.:   --γ: {dependency(heart_disc, liste, decision_attr)}   --Anzahl an Regeln: {len(induce_rules(heart_disc, liste, decision_attr))}")
    print(10*'--')

1.:   --γ: 1.0   --Anzahl an Regeln: 290
--------------------
2.:   --γ: 0.9866220735785953   --Anzahl an Regeln: 285
--------------------
3.:   --γ: 0.9866220735785953   --Anzahl an Regeln: 267
--------------------
4.:   --γ: 0.9866220735785953   --Anzahl an Regeln: 267
--------------------
5.:   --γ: 0.979933110367893   --Anzahl an Regeln: 264
--------------------
6.:   --γ: 0.9765886287625418   --Anzahl an Regeln: 261
--------------------
7.:   --γ: 0.9331103678929766   --Anzahl an Regeln: 235
--------------------
8.:   --γ: 0.7324414715719063   --Anzahl an Regeln: 94
--------------------
9.:   --γ: 0.9364548494983278   --Anzahl an Regeln: 249
--------------------
10.:   --γ: 0.979933110367893   --Anzahl an Regeln: 258
--------------------
